In [ ]:
"""
PINN inverse problem: infer the repulsive coupling alpha in  F = alpha / (|a0 + x2 - x1| + thickness)^4
from the motion of two spring-bound magnets (mass m, spring k).  Colab-ready, CPU is fine (tiny problem).

Edit the CONFIG block only.
"""
import time, numpy as np, pandas as pd, torch, torch.nn as nn
torch.set_default_dtype(torch.float64)       # alpha is a ~1% effect on the dynamics -> use double precision
torch.set_num_threads(1)

# ===================== CONFIG =====================
path   = '/content/magnetic_quasiparticle_dataset (8).xlsx'   # <- your uploaded file
seed   = 0
USE_V  = 1               # 1: also fit the velocity columns (strongly recommended), 0: positions only
N1, N2, NL = 3000, 4000, 300   # Adam data-only steps, Adam data+physics steps, L-BFGS iterations
# ==================================================
torch.manual_seed(seed); np.random.seed(seed)

# ---------- 1. read header block + table (no hand-typed constants) ----------
raw = pd.read_excel(path, header=None)
par = {str(raw.iloc[i, 0]).split('(')[0].strip(): float(str(raw.iloc[i, 1]).split()[0]) for i in range(1, 6)}
m, k = par['Mass'], par['Spring Stiffness']
a0, th, alpha_true = par['Equilibrium Lattice Gap'], par['Magnet Thickness'], par['True Coupling Constant']
r0 = raw.index[raw.iloc[:, 0].astype(str).str.contains('Time')][0]
t_np, x1_np, x2_np, v1_np, v2_np = raw.iloc[r0 + 1:].astype(float).values.T
T = t_np[-1]
print(f"m={m} k={k} a0={a0} thickness={th} alpha_true={alpha_true} N={len(t_np)} T={T}")

# noise levels straight from the data (x1+x2 and v1+v2 should vanish for the antisymmetric motion)
sx = (x1_np + x2_np).std() / np.sqrt(2); sv = (v1_np + v2_np).std() / np.sqrt(2)
A = np.sqrt(2) * x1_np.std()                 # amplitude scale, only used to make the network output O(1)
w0 = np.sqrt(k / m)
# dominant frequency from the data (zero-padded FFT of the relative coordinate)
rel = (x2_np - x1_np); rel = (rel - rel.mean()) * np.hanning(len(rel))
F = np.abs(np.fft.rfft(rel, 1 << 16)); fr = np.fft.rfftfreq(1 << 16, t_np[1] - t_np[0])
f_pk = fr[F.argmax()]
print(f"noise: sigma_x={sx:.4f} sigma_v={sv:.4f}; amplitude scale A={A:.3f}; FFT peak {f_pk:.3f} Hz (sqrt(k/m)/2pi={w0/2/np.pi:.3f} Hz)")

td = torch.tensor(t_np).view(-1, 1)
X1, X2, V1, V2 = (torch.tensor(a).view(-1, 1) for a in (x1_np, x2_np, v1_np, v2_np))
Nd = len(t_np)

# ---------- 2. network: slowly varying envelopes on top of oscillatory (Fourier) features ----------
class PINN(nn.Module):
    def __init__(s, f0, H=64):
        super().__init__()
        s.f0 = f0
        s.dw = nn.Parameter(torch.zeros(1))                     # relative correction of carrier frequency
        s.mlp = nn.Sequential(nn.Linear(7, H), nn.Tanh(), nn.Linear(H, H), nn.Tanh(),
                              nn.Linear(H, H), nn.Tanh(), nn.Linear(H, 2))
        s.log_alpha = nn.Parameter(torch.log(torch.tensor([0.01])))   # alpha = exp(.) > 0, starts at 0.01 like yours
    @property
    def alpha(s): return torch.exp(s.log_alpha)
    def forward(s, t):
        ph = 2 * np.pi * s.f0 * (1 + s.dw) * t
        z = torch.cat([t / T, torch.sin(ph), torch.cos(ph), torch.sin(2 * ph), torch.cos(2 * ph),
                       torch.sin(3 * ph), torch.cos(3 * ph)], 1)
        return A * s.mlp(z)

net = PINN(f_pk)

def fields(t):
    t = t.clone().requires_grad_(True)
    out = net(t)
    x1, x2 = out[:, :1], out[:, 1:]
    g = lambda y, tt: torch.autograd.grad(y, tt, torch.ones_like(y), create_graph=True)[0]
    v1, v2 = g(x1, t), g(x2, t)
    a1, a2 = g(v1, t), g(v2, t)
    return x1, x2, v1, v2, a1, a2

def data_loss(x1, x2, v1, v2):
    lx = ((x1[:Nd] - X1) ** 2 + (x2[:Nd] - X2) ** 2).mean() / sx ** 2 / 2
    lv = ((v1[:Nd] - V1) ** 2 + (v2[:Nd] - V2) ** 2).mean() / sv ** 2 / 2
    return lx, lv

def phys_loss(x1, x2, a1, a2, sig_r):
    f = net.alpha / ((torch.abs(a0 + x2 - x1) + th) ** 4)
    r1 = m * a1 + k * x1 + f
    r2 = m * a2 + k * x2 - f
    return ((r1 ** 2).mean() + (r2 ** 2).mean()) / (2 * sig_r ** 2)

def report(tag, lx, lv, lp, ep):
    print(f"{tag} ep {ep:5d} | chi2/pt  x:{lx.item():8.3f} v:{lv.item():8.3f} | phys {lp:10.3f} | alpha {net.alpha.item():.5f} | dw {net.dw.item()*100:+.3f}%", flush=True)

t0 = time.time()
# ---------- Stage 1: data only (find the trajectory first) ----------
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
for ep in range(1, N1 + 1):
    opt.zero_grad()
    x1, x2, v1, v2, _, _ = fields(td)
    lx, lv = data_loss(x1, x2, v1, v2)
    (lx + (lv if USE_V else 0)).backward(); opt.step()
    if ep % 1000 == 0 or ep == N1: report('S1 data  ', lx, lv, 0.0, ep)

# ---------- Stage 2: add physics, tighten the tolerance sigma_r geometrically ----------
opt = torch.optim.Adam([{'params': net.mlp.parameters(), 'lr': 1e-3},
                        {'params': [net.dw], 'lr': 2e-4},
                        {'params': [net.log_alpha], 'lr': 1e-2}])
sig_hi, sig_lo = 2.0, 0.01
for ep in range(1, N2 + 1):
    sig_r = sig_hi * (sig_lo / sig_hi) ** (ep / N2)
    tc = torch.cat([td, torch.rand(400, 1) * T])
    opt.zero_grad()
    x1, x2, v1, v2, a1, a2 = fields(tc)
    lx, lv = data_loss(x1, x2, v1, v2)
    lp = phys_loss(x1, x2, a1, a2, sig_r)
    (lx + (lv if USE_V else 0) + lp).backward(); opt.step()
    if ep % 1000 == 0 or ep == N2: report('S2 +phys ', lx, lv, lp.item(), ep)

# ---------- Stage 3: L-BFGS polish, fixed collocation grid, tight physics ----------
tc = torch.cat([td, torch.linspace(0, T, 600).view(-1, 1)])
lb = torch.optim.LBFGS(net.parameters(), lr=1, max_iter=NL, history_size=50, line_search_fn='strong_wolfe',
                       tolerance_grad=1e-12, tolerance_change=1e-14)
def closure():
    lb.zero_grad()
    x1, x2, v1, v2, a1, a2 = fields(tc)
    lx, lv = data_loss(x1, x2, v1, v2)
    lp = phys_loss(x1, x2, a1, a2, sig_lo)
    L = lx + (lv if USE_V else 0) + lp
    L.backward(); return L
lb.step(closure)
x1, x2, v1, v2, a1, a2 = fields(tc)
lx, lv = data_loss(x1, x2, v1, v2); lp = phys_loss(x1, x2, a1, a2, sig_lo).item()
report('S3 LBFGS ', lx, lv, lp, NL)
a = net.alpha.item()
print(f"RESULT seed={seed} use_v={USE_V}: alpha = {a:.4f}  (stated true {alpha_true}, rel.err vs stated {100*(a/alpha_true-1):+.1f}%)  [{time.time()-t0:.0f}s]")

m=0.3 k=50.0 a0=1.0 thickness=0.05 alpha_true=0.5 N=300 T=3.0
noise: sigma_x=0.0192 sigma_v=0.0193; amplitude scale A=0.114; FFT peak 2.123 Hz (sqrt(k/m)/2pi=2.055 Hz)
S1 data   ep  1000 | chi2/pt  x:   1.016 v:   0.871 | phys      0.000 | alpha 0.01000 | dw -0.025%
S1 data   ep  2000 | chi2/pt  x:   1.015 v:   0.832 | phys      0.000 | alpha 0.01000 | dw -0.025%
S1 data   ep  3000 | chi2/pt  x:   1.014 v:   0.814 | phys      0.000 | alpha 0.01000 | dw -0.021%
S2 +phys  ep  1000 | chi2/pt  x:   1.019 v:   0.840 | phys      0.056 | alpha 0.49231 | dw -0.016%
S2 +phys  ep  2000 | chi2/pt  x:   1.024 v:   1.051 | phys      0.350 | alpha 0.50201 | dw -0.014%
S2 +phys  ep  3000 | chi2/pt  x:   1.026 v:   0.976 | phys      0.376 | alpha 0.50349 | dw -0.017%
S2 +phys  ep  4000 | chi2/pt  x:   1.029 v:   1.018 | phys      8.927 | alpha 0.50302 | dw -0.018%
S3 LBFGS  ep   300 | chi2/pt  x:   1.025 v:   0.981 | phys      0.098 | alpha 0.50243 | dw -0.017%
RESULT seed=0 use_v=1: alpha = 0.5024  (

In [ ]:
from google.colab import drive
drive.mount('/content/drive')